# Exercise 02 — read what you produced

**Goal:** start a Kafka consumer, read the events you sent in Exercise 01, and observe how *consumer groups* let Kafka remember your read position.

**Prerequisite:** you ran [`exercise_01_produce_single.ipynb`](exercise_01_produce_single.ipynb) and there are events in `strom` and `wasser`.

## Background — how consumers track progress

Unlike a queue, Kafka does **not** delete messages after they are read. Every consumer can read every message — possibly multiple times. What is *consumer-specific* is the **offset**: how far each consumer *group* has already read, per partition.

- **`group.id`** — the name of *your* consumer group. Two notebooks   with the same group share the work; with different groups they each   see all messages independently.
- **`auto.offset.reset`** — only used when *no* offset is stored yet   (first start). `earliest` = read all history; `latest` = only new.

## Step 1 — configure

**Task — pick a `group.id`.** Any string works; convention is `<app>-<purpose>` so it shows up well in monitoring tools.

In [ ]:
from confluent_kafka import Consumer

conf = {
    'bootstrap.servers': 'redpanda:29092',
    'group.id':          None,            # TODO: e.g. 'energy-monitor'
    'auto.offset.reset': 'earliest',      # read history if no offset stored
}
consumer = Consumer(conf)
print('Consumer created.')

## Step 2 — subscribe

`subscribe()` does *not* immediately read anything. It tells the broker which topics you're interested in. The actual partition assignment happens on the first `poll()`.

**Task — subscribe to both `strom` and `wasser`.**

In [ ]:
# TODO: consumer.subscribe(['strom', 'wasser'])

print('Subscribed.')

## Step 3 — poll for messages

`consumer.poll(timeout)` is the heart of every Kafka consumer:

1. If the broker has new messages, return one.
2. Otherwise, wait up to `timeout` seconds and return `None`.
3. While waiting, send heartbeats to the broker so it knows we're alive.

We use a counter to stop after a few empty polls — otherwise this loop would run forever (which is normal for production consumers).

In [ ]:
messages_read = 0
empty_polls   = 0

while messages_read < 20 and empty_polls < 5:
    msg = consumer.poll(2.0)
    if msg is None:
        empty_polls += 1
        continue
    if msg.error():
        print(f'Error: {msg.error()}')
        continue

    empty_polls    = 0
    messages_read += 1
    key   = msg.key().decode()   if msg.key()   else 'None'
    value = msg.value().decode() if msg.value() else 'None'
    print(f'[{messages_read}] {msg.topic()} P{msg.partition()} '
          f'offset={msg.offset()}')
    print(f'     key={key}  value={value}')

print(f'Read {messages_read} message(s).')

**Take a moment to look at the output.** For every message you should see: which **topic** it came from, which **partition** stored it, the **offset** within that partition, and the **key** + **value** the producer sent. Notice that offsets are independent per partition: partition 0 has its own 0,1,2,… and partition 1 also starts at 0.

## Task A — live streaming

The cell below polls for up to 30 seconds. Run it, then in a *separate tab* open the producer notebook and send a new event. **It should appear here within ~1 second** — that's the latency of Kafka in this small setup.

Tip: drag the producer notebook tab to the right edge of VS Code to split the editor and see both notebooks at once.

In [ ]:
from datetime import datetime

print('Waiting for new events (30s)...')
empty_polls = 0
while empty_polls < 15:   # 15 × 2s = 30s
    msg = consumer.poll(2.0)
    if msg is None:
        empty_polls += 1; continue
    if msg.error():
        continue
    empty_polls = 0
    ts  = datetime.now().strftime('%H:%M:%S')
    key = msg.key().decode() if msg.key() else 'None'
    print(f'[{ts}] {msg.topic()} P{msg.partition()} '
          f'offset={msg.offset()} key={key}')
    print(f'       {msg.value().decode()}')

print('No new messages.')

## Task B — different group, same data

Below we make a *new* consumer with a *different* `group.id` and read the topic again. Even though we already "read" the data above, the new group has no stored offset → it starts from the beginning.

**This is fundamental.** Kafka's design encourages many independent consumer groups — one for the dashboards, one for the alerting service, one for the data-warehouse loader. Each pays the storage cost zero times (Kafka stores the data once) but reads it at its own pace.

In [ ]:
consumer.close()

# TODO: create a new Consumer with group.id='analytics' and
#       auto.offset.reset='earliest'; subscribe to ['strom', 'wasser'];
#       read up to 10 events. Notice that you re-read all events.


In [ ]:
try:
    consumer.close()  # release the partition assignment cleanly
except Exception:
    pass
print('Cleaned up.')

## What you learned

- A consumer connects, **subscribes** to topics and **polls** for   messages.
- Kafka tracks read progress per *consumer group*, not per individual   consumer.
- `auto.offset.reset` only kicks in for brand-new groups.
- Different groups read the same data **independently** — that's how   Kafka supports many readers without duplicating storage.

Next: see how partitions and keys interact with consumer groups → [`exercise_03_produce_batch.ipynb`](exercise_03_produce_batch.ipynb).